# Task 1: Working with data in Python

In this unit we will focus on how to deal with data in Python. The first part involves elementary data management operations (e.g., import, conversion, output). In the second we will focus on plotting data. 

In [ ]:
import pandas as pd

## Part 1: Data mingling

### 1.1 Construct complete data sets for Chicago and Potsdam for 2015

#### 1.1.1 Chicago

#### Import the data of Chicago based on local time from .csv (01/01/2015-12/31/2015)

In [ ]:
# Read the CSV file
CH_15_df = pd.read_csv("data/PV_generation_Chicago_10kW_0loss_2015.csv", 
                       usecols=[1,2,5],  # what is happening here?
                       skiprows=3,  # what is happening here?
                       parse_dates=['local_time'],  # what is happening here?
                       encoding='latin1')  # what is happening here?

# what is happening here?
CH_15_df.head(10)

All there?

In [ ]:
CH_15_df.shape 
# correct output: 8760 rows and 3 columns

In [ ]:
CH_15_df.tail() # print last 5 rows

>Inspect the imported data. What do you notice? What is yet to be done to compile a complete data set of 2015?

#### Modify data set to include all data points for 2015

In [ ]:
# Import data of 2016 into Dataframe CH_16_df
#CH_16_df = ...

#CH_16_df.head()

In [ ]:
# What is happening here?
CH_df = pd.concat([CH_15_df.iloc[6:], CH_16_df.iloc[:6]], ignore_index=True)

# 'reset_index': index aligns with the new concatenated dataframe

# Display the last 10 rows
CH_df.tail(10)

#### 1.1.2 Potsdam 

#### Import the data of Potsdam based on local time from .csv (01/01/2015-12/31/2015)

>Take a look at the .csv file. What do you notice?

In [ ]:
# Read the CSV file
P_15_df = pd.read_csv(
    "data/PV_generation_Potsdam_10kW_0loss_2015_GermanFormat.csv", 
    usecols=[1, 2, 5],  # Select columns 2, 3, 6 (0-indexed in Python)
    skiprows=3,  # Skip the first 3 rows (header starts at the 4th row in Python)
    delimiter=';',  # what is happening here?
    decimal=',',  # what is happening here?
    parse_dates=['local_time'], # Determine where to find the time data
    date_parser=lambda x: pd.to_datetime(x, format="%d.%m.%Y %H:%M") # Trick to cope with the german date format
)

# Display the first 5 rows of the DataFrame
P_15_df.head(5)

All there?

In [ ]:
P_15_df.shape
# correct output: 8760 rows and 3 columns

In [ ]:
P_15_df.tail()

#### Modify data set to include all data points for 2015

In [ ]:
#Import data of 2014 into Dataframe CH_14_df
#P_14_df = ...

#P_14_df.tail(10)

In [ ]:
# Create DataFrame P_df that holds all data points for 2015
#P_df = ...

# Display the last 10 rows
P_df.tail(10)

In [ ]:
P_df.head()

Are `CH_df` and `P_df` the same size?

In [ ]:
CH_df.shape, P_df.shape

### 1.2 Merge both datasets and export to .csv file

In [ ]:
# what is happening here?
df = pd.merge(CH_df, P_df, on='local_time', how='inner')

# Display the first 5 rows
df.head()

In [ ]:
df.columns = ['time', 'CH_PV', 'CH_temp', 'P_PV', 'P_temp']  # what is happening here?

df.head()  # display the first 5 rows

In [ ]:
df.shape

Create or overwrite a .csv file at `data/pv_temp_both.csv` with `to_csv`


In [ ]:
df.to_csv("data/pv_temp_both.csv", index=False) # 'index=False' excludes the row index from being written to the CSV file.

>#### What's the issue when concatenating both DataFrames?

### 1.3. Calculate monthly mean values

Add column with month indicator

In [ ]:
df['month'] = df['time'].dt.month # only possible, when "df['time']" is a DateTime Obj.

# what is happening here?

In [ ]:
df.head()

In [ ]:
df.columns[1] # get name of column 2 (0-indexed)

First, let's create a new DataFrame `df_mean` where you can find the mean for `CH_PV` for each month.

In [ ]:
# Define new DataFrame that holds monthly mean values for CH_PV_mean

# df_mean = 

# df_mean # Display df_mean

Rename the column to `CH_PV_mean`

In [ ]:
# Rename the column
df_mean.rename(columns={"CH_PV": "CH_PV_mean"}, inplace=True) # inplace=True modifies the DataFrame df_mean directly, without the need to assign the result back to a new variable.

# Display the DataFrame
df_mean

Now, let's overwrite and rename the DataFrame `df_mean` so you can find the mean values for `'CH_PV', 'CH_temp', 'P_PV', 'P_temp'` for each month. (same procedure)

In [ ]:
# Define new DataFrame based on aggregation by month
df_mean = df.groupby('month')[['CH_PV', 'CH_temp', 'P_PV', 'P_temp']].mean().reset_index()

# what is happening here?
df_mean.rename(columns={
    "CH_PV": "CH_PV_mean", 
    "CH_temp": "CH_temp_mean",
    "P_PV": "P_PV_mean",
    "P_temp": "P_temp_mean"}, inplace=True)

# Display the DataFrame
df_mean

We can transform the DataFrame back to a matrix (= 2D NumPy Array) with `.to_numpy()`

In [ ]:
matrix_df_mean = df_mean.to_numpy() # convert a DataFrame to a numpy array

matrix_df_mean

### 1.4 Save to .csv file

In [ ]:
df_mean.to_csv("data/pv_temp_monthly_mean.csv", index=False)

# 'index=False' prevents pandas from writing row indices into the CSV file

Break ☕😎

## Part 2: Visualization

In [ ]:
#In case you haven't done so, let's import our libraries

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### 2.1 Visualize monthly mean values

In [ ]:
df_mean = pd.read_csv("data/pv_temp_monthly_mean.csv") # Import data from .csv into DataFrame (if no longer available from Task 1)

In [ ]:
df_mean # print result

#### Let's create a combined chart of the PV generation and temperature of each month

In [ ]:
x = np.array(range(1,13)) # Create x values (months)
x # display

In [ ]:
# Extract PV and temperature values
PV_mean = df_mean[['CH_PV_mean', 'P_PV_mean']].to_numpy() # what is happening here?
T_mean = df_mean[['CH_temp_mean', 'P_temp_mean']].to_numpy() # what is happening here?

In [ ]:
# Import the necessary library
import matplotlib.pyplot as plt

# what is happening here?
fig, ax1 = plt.subplots(figsize=(7,5)) 

# First Axis: Bar Chart for PV Generation
# what is happening here?
ax1.bar(x - 0.2, PV_mean[:, 0], 0.4, label='CH_PV', color='paleturquoise', alpha=0.8)
ax1.bar(x + 0.2, PV_mean[:, 1], 0.4, label='P_PV', color='palegoldenrod', alpha=0.8)

# Labelling x-axis as 'Months' and y-axis as 'PV generation [kW]'
ax1.set_xlabel('Months')
ax1.set_ylabel('PV generation [kW]')

# Setting the limits for the x-axis and y-axis manually
ax1.set_xlim(0, 13)
ax1.set_ylim(0, 3)

# Setting the x-axis ticks to match the month numbers.
ax1.set_xticks(x)

# Second Axis: Line Graph for Temperature
# what is happening here?
ax2 = ax1.twinx()

# what is happening here?
ax2.plot(x, T_mean[:, 0], color='darkcyan', linewidth=2, marker='o', label='CH_Temp')
ax2.plot(x, T_mean[:, 1], color='goldenrod', linewidth=2, marker='o', label='P_Temp')

# Labelling and setting the limits for the second y-axis
ax2.set_ylabel('Temperature [°C]')
ax2.set_ylim(-10, 30)

# what is happening here?
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper left')

# Display the entire Plot
plt.show()

In [ ]:
fig.savefig("figures/PV_temp_monthly_means.pdf")
# or as PNG: fig.savefig("figures/PV_temp_monthly_means.png")

### 2.2 Visualize confidence intervals

Load the initial merged DataFrame and add a column for the hour

In [ ]:
# Read the CSV file into a DataFrame
df = pd.read_csv("data/pv_temp_both.csv", parse_dates=['time'])

# Add a new column 'hour' with information on the hour of the day
df['hour'] = df['time'].dt.hour

# Display the first 5 rows of the DataFrame
df.head()

Chicago DataFrame with median values

In [ ]:
# Compute median values for Chicago PV generation and save to new DataFrame df_CH

# df_CH = 

In [ ]:
df_CH

Now the complete Chicago DataFrame with median values and confidence intervals

In [ ]:
# PV quantiles
df_CH = df.groupby('hour')['CH_PV'].agg(['median', lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)]).reset_index()
df_CH.columns = ['hour', 'CH_PV_median', 'CH_PV_25_quant', 'CH_PV_75_quant']

# add temperature indicators
df_temp = df.groupby('hour')['CH_temp'].agg(['median', lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)]).reset_index()
df_temp.columns = ['hour', 'CH_Temp_median', 'CH_Temp_25_quant', 'CH_Temp_75_quant']

# join two dataframes
df_CH = pd.merge(df_CH, df_temp, on='hour')

df_CH

Now for Potsdam: DataFrame with median values and quantiles

In [ ]:
# PV quantiles
df_P = df.groupby('hour')['P_PV'].agg(['median', lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)]).reset_index()
df_P.columns = ['hour', 'P_PV_median', 'P_PV_25_quant', 'P_PV_75_quant']

# Temperature quantiles
df_temp = df.groupby('hour')['P_temp'].agg(['median', lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)]).reset_index()
df_temp.columns = ['hour', 'P_Temp_median', 'P_Temp_25_quant', 'P_Temp_75_quant']

# join two dataframes
df_P = pd.merge(df_P, df_temp, on='hour')

df_P

In [ ]:
# what is happening here?
h = df_CH['hour'].values
y_CH = df_CH.drop('hour', axis=1).values
y_P = df_P.drop('hour', axis=1).values

Create visualization

In [ ]:
import matplotlib.pyplot as plt

# Set common limits of y-axis to compare the plots
PV_ymax = 9
Temp_ymax = 26

# Initialize a 2x2 subplot figure
fig, axs = plt.subplots(2, 2, figsize=(12,8)) # use 'axs' when creating a figure with multiple subplots

# Chicago PV
axs[0, 0].fill_between(h, y_CH[:,2], y_CH[:,1], color='gold', alpha=0.3)
axs[0, 0].plot(h, y_CH[:,0], color='gold', linewidth=4.0)
axs[0, 0].set_ylim(0, PV_ymax)
axs[0, 0].set_title('Chicago PV')
axs[0, 0].set_ylabel('PV generation [kW]')
axs[0, 0].set_xlabel('Hour')

# Potsdam PV
axs[0, 1].fill_between(h, y_P[:,2], y_P[:,1], color='gold', alpha=0.3)
axs[0, 1].plot(h, y_P[:,0], color='gold', linewidth=4.0)
axs[0, 1].set_ylim(0, PV_ymax)
axs[0, 1].set_title('Potsdam PV')
axs[0, 1].set_ylabel('PV generation [kW]')
axs[0, 1].set_xlabel('Hour')

# Chicago Temperature
axs[1, 0].fill_between(h, y_CH[:,5], y_CH[:,4], color='paleturquoise', alpha=0.3)
axs[1, 0].plot(h, y_CH[:,3], color='paleturquoise', linewidth=4.0)
axs[1, 0].set_ylim(0, Temp_ymax)
axs[1, 0].set_ylabel('Temperature [°C]')
axs[1, 0].set_title('Chicago Temperature')
axs[1, 0].set_xlabel('Hour')

# Potsdam Temperature
axs[1, 1].fill_between(h, y_P[:,5], y_P[:,4], color='paleturquoise', alpha=0.3)
axs[1, 1].plot(h, y_P[:,3], color='paleturquoise', linewidth=4.0)
axs[1, 1].set_ylim(0, Temp_ymax)
axs[1, 1].set_ylabel('Temperature [°C]')
axs[1, 1].set_title('Potsdam Temperature')
axs[1, 1].set_xlabel('Hour')

# Adjust layout for neat appearance
plt.tight_layout()
plt.show()


In [ ]:
# save figure
fig.savefig('figures/PV_temp_daily_confidence.pdf')

done 👍